Personal Chef Project.

In [1]:
# first things first is always to import load_dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Create the necessary tools
# We need the langchain tools library
from langchain.tools import tool

# We also need to import the tavily module as well as dict, any
from typing import Dict,Any
from tavily import TavilyClient

tavily_client = TavilyClient() # no idea what this does

@tool
def web_search(query: str) -> Dict[str,Any]:
    """Search the web for information"""
    return tavily_client.search(query)


In [3]:
# Include the option to upload an image of the fridge or pantry.
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [4]:
print(uploader.value)

({'name': 'fridge.png', 'type': 'image/png', 'size': 437931, 'content': <memory at 0x0000020C7D2434C0>, 'last_modified': datetime.datetime(2026, 1, 20, 18, 4, 41, 348000, tzinfo=datetime.timezone.utc)},)


In [5]:
#Conversion into base 64
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
# Create Langchain agent. (Gemini 2.5 flash lite is the model used)
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver # to allow for a thread id enabling followup questions.

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

agent = create_agent(model= model,
    system_prompt="You are a personal chef. Using the ingredients provided, search for or provide a recipe to create a decent 1 serving meal.Use the web search tool to search the web for recipes using ingredients present.",
    checkpointer=InMemorySaver(),
    tools=[web_search]
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What food can i make for lunch?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

You have a roasted chicken, tomatoes, bell peppers, and eggs. Here's a recipe for a simple and satisfying lunch:

**Chicken and Vegetable Scramble**

This recipe uses the roasted chicken and some of the vegetables for a quick and flavorful scramble.

**Ingredients:**

*   About 1/2 cup cooked roasted chicken, shredded or diced
*   1/4 cup diced tomatoes
*   1/4 cup diced bell pepper (any color)
*   1-2 eggs
*   Salt and pepper to taste
*   A small amount of oil or butter for cooking (if needed, depending on how the chicken was roasted)

**Instructions:**

1.  **Prepare the chicken and vegetables:** If your chicken is not already shredded or diced, do so now. Dice the tomatoes and bell pepper.
2.  **Sauté vegetables (optional):** If you prefer your vegetables slightly softened, heat a small amount of oil or butter in a non-stick skillet over medium heat. Add the diced bell pepper and cook for 2-3 minutes until slightly tender. Then add the diced tomatoes and cook for another minute.
3. 

In [8]:
from pprint import pprint

pprint(response)


{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'What food can i make for lunch?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='dcb8f21a-5262-4d39-add3-dd7808484f2a'),
              AIMessage(content="You have a roasted chicken, tomatoes, bell peppers, and eggs. Here's a recipe for a simple and satisfying lunch:\n\n**Chicken and Vegetable Scramble**\n\nThis recipe uses the roasted chicken and some of the vegetables for a quick and flavorful scramble.\n\n**Ingredients:**\n\n*   About 1/2 cup cooked roasted chicken, shredded or diced\n*   1/4 cup diced tomatoes\n*   1/4 cup diced bell pepper (any color)\n*   1-2 eggs\n*   Salt and pepper to taste\n*   A small amount of oil or butter for cooking (if needed, depending on how the chicken was roasted)\n\n**Instructions:**\n\n1.  **Prepare the chicken and vegetables:** If your chicken is not already shredded or diced, do so now. Dice the tomatoes and bell pepper.\

In [10]:
# For follow up
question = HumanMessage(content="I feel like stuffed bell peppers instead")

response = agent.invoke(
    {"messages": [question]},
    config
)
print(response['messages'][-1].content)

That's a great idea! Stuffed bell peppers are delicious. Given the ingredients you have, here's a recipe for stuffed bell peppers. Since you have whole bell peppers and roasted chicken, we can adapt the classic recipe.

**Roasted Chicken and Tomato Stuffed Bell Peppers**

This recipe uses your roasted chicken as the base for the stuffing, and we'll incorporate some of the tomatoes.

**Ingredients:**

*   1 large bell pepper (any color from your fridge)
*   About 1 cup of your roasted chicken, shredded or finely diced
*   1/4 cup diced tomatoes (or more, depending on how much you want)
*   1/4 cup cooked rice or quinoa (if you have any leftovers, otherwise, this can be omitted or you can cook a small amount)
*   1-2 tablespoons of chopped herbs like parsley (if available)
*   Salt and pepper to taste
*   A little bit of olive oil or butter (optional, for sautéing)
*   A splash of water or broth (optional, for steaming)

**Instructions:**

1.  **Prepare the Bell Pepper:**
    *   Cut the

In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'What food can i make for lunch?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='dcb8f21a-5262-4d39-add3-dd7808484f2a'),
              AIMessage(content="You have a roasted chicken, tomatoes, bell peppers, and eggs. Here's a recipe for a simple and satisfying lunch:\n\n**Chicken and Vegetable Scramble**\n\nThis recipe uses the roasted chicken and some of the vegetables for a quick and flavorful scramble.\n\n**Ingredients:**\n\n*   About 1/2 cup cooked roasted chicken, shredded or diced\n*   1/4 cup diced tomatoes\n*   1/4 cup diced bell pepper (any color)\n*   1-2 eggs\n*   Salt and pepper to taste\n*   A small amount of oil or butter for cooking (if needed, depending on how the chicken was roasted)\n\n**Instructions:**\n\n1.  **Prepare the chicken and vegetables:** If your chicken is not already shredded or diced, do so now. Dice the tomatoes and bell pepper.\

In [ ]:
# For follow up
question = HumanMessage(content="How does the original recipe differ to mine?")

response = agent.invoke(
    {"messages": [question]},
    config
)
print(response['messages'][-1].content)

The concept of stuffed bell peppers is believed to have originated in Central and South America, following the introduction of bell peppers from the Americas to Europe. Different cultures around the world have their own unique versions of this dish.


In [17]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Where does the recipe originate from?', additional_kwargs={}, response_metadata={}, id='43ea4b06-9d55-4c3c-ad2b-8bbe2d9ac656'),
              AIMessage(content='I need to know which recipe you are referring to. Please provide me with the name of the recipe.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bdca6-75c4-75f1-8246-ac35f9b5c969-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 21, 'total_tokens': 107, 'input_token_details': {'cache_read': 0}}),
              HumanMessage(content='Stuffed Bell peppers', additional_kwargs={}, response_metadata={}, id='711f31f2-40c8-4d82-a545-b839663f04ad'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "Stuffed Bell peppers origin"}'}}, response_metadata={'finish_r